In [ ]:
!pip install transformers pdf2image faiss-cpu

In [ ]:
!apt-get install -y poppler-utils

In [ ]:
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
import torch
import faiss
from pdf2image import convert_from_path
import numpy as np
from PIL import Image
from IPython.display import Image, display
import os
from transformers import ColQwen2ForRetrieval, ColQwen2Processor
from transformers.utils.import_utils import is_flash_attn_2_available
import base64
import requests
from google.colab import userdata
import gc
import io

In [ ]:
#------Encoder-Model-------!
encoder_model = ColQwen2ForRetrieval.from_pretrained('vidore/colqwen2-v1.0-hf',device_map='cuda:0',attn_implementation='flash_attention_2' if is_flash_attn_2_available() else 'sdpa')
processor = ColQwen2Processor.from_pretrained('vidore/colqwen2-v1.0-hf')

In [ ]:
#----Decoder-Model-------!

In [ ]:
#----paths-----!
pdf_path = '/kaggle/input/datasets/mihirvyas/attentionisallyouneed/NIPS-2017-attention-is-all-you-need-Paper.pdf'
output_folder = 'extracted_pdf_pages'

In [ ]:
def page_to_image(pdf_path,dpi,fmt,output_folder):
  #-----Create directory if not exist----!
  if not os.path.exists(output_folder):
    os.makedirs(output_folder)

  #-----convert pages to image-----!
  images = convert_from_path(pdf_path=pdf_path,dpi=dpi,output_folder=output_folder,fmt=fmt,transparent=False)
  return images

In [ ]:
images = page_to_image(pdf_path=pdf_path,dpi=150,fmt='png',output_folder=output_folder)

In [ ]:
def process_embedImages(pages,model,proc,batch_size=2) -> list[torch.Tensor]:
  all_embeddings = []

  for i in range(0,len(pages),batch_size):
    batch = pages[i:i+batch_size]
    inputs = proc(images=batch).to(model.device)

    with torch.no_grad():
      embedding = model(**inputs).embeddings

    print(f'Embedding Shape: {embedding.shape}')

    #Move to cpu
    for emb in embedding:
      all_embeddings.append(emb.cpu().detach())
  print(len(all_embeddings))
  return all_embeddings

In [ ]:
%%time
embedded = process_embedImages(pages=images,model=encoder_model,proc=processor,batch_size=2)

# **First Approach: Faiss**

In [ ]:
page_embedded = torch.stack([emb.mean(dim=0) for emb in embedded])
page_embedded.shape

In [ ]:
type(page_embedded)

In [ ]:
#-----Faiss-----!
page_embedded_np = page_embedded.float().numpy()

faiss.normalize_L2(page_embedded_np)

dim = page_embedded_np.shape[1]

index = faiss.IndexFlatIP(dim)

index.add(page_embedded_np)

In [ ]:
def retrieveFaiss(query,top_k=3):
  input_text = processor(text=[query]).to(encoder_model.device)

  with torch.no_grad():
    embeddings = encoder_model(**input_text).embeddings

  query_embedding = embeddings.mean(dim=1) # here dim 1 use because query is token based. so we get average based on token
  query_np = (query_embedding.float().cpu().numpy())
  faiss.normalize_L2(query_np)

  scores,indices = index.search(query_np,top_k)
  results = []

  for score, idx in zip(scores[0], indices[0]):
      results.append({"page_index": int(idx),"score": float(score)})

  return results

# **Second Approach: Native**

In [ ]:
image_embeddings = torch.stack(embedded)

In [ ]:
def retrieveNative(query,top_k=3):
  input_text = processor(text=[query]).to(encoder_model.device)

  with torch.no_grad():
    query_embeddings = encoder_model(**input_text).embeddings

  print(query_embeddings.shape)
  scores = processor.score_retrieval(query_embeddings,image_embeddings.to(query_embeddings.device))
  top_scores, top_indices = torch.topk(scores[0],k=3)

  fresults = []
  print('Native:')
  for idx, score in zip(top_indices, top_scores):
    print(f"Page: {idx.item()} | Score: {score.item():.4f}")
    fresults.append({"page_index": int(idx.item()),"score": float(score.item())})

  return fresults

In [ ]:
questions = [
    "What model architecture is in the paper?",
    "What does the encoder in Transformer consist of?",
    "How many attention heads did they use?",
    "What BLEU score did they achieve on English-German translation?",
    "How long did training take on 8 GPUs?",
    "What is the key advantage over RNN models?",
    "What optimizer did they use?",
    "What is multi-head attention?"
]

# **LLM**

In [ ]:
def RetriveAnswer_From_LLM(image,question,model,API_KEY,url):
  #-----Header----@
  headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
  }

  #--Convert image to base64--!
  buffer = io.BytesIO()
  image.save(buffer, format="PNG")
  image_b64 = base64.b64encode(buffer.getvalue()).decode()

  #-----payload-----@
  payload = {
    "model":model,
    "messages":[
        {
            "role":"system",
            "content":"You are a helpful question-answering assistant."
        },
        {
            "role":"user",
            "content":[
                {
                    "type":"image_url",
                    "image_url": {"url":f"data:image/png;base64,{image_b64}"}
                },
                {
                    "type":"text",
                    "text":f""" Answer the question using only the information visible in the provided image.
                     Quesiton:
                     {question}

                     If the answer is not visible in the image, say:
                     'I cannot find an answer on the provided docs.'
                     """
                }
            ]
        }
    ],
    "max_tokens":80,
    "temperature":0.2,
    "stream":False
  }

  #-----Get Response----!
  request = requests.post(url=url,headers=headers,json=payload)
  response = request.json()

  if request.status_code != 200:
    return f"Request Failed ({request.status_code}): {response}"

  if "error" in response:
    return f"API Error: {response['error'].get('message')}"

  return (
    response
    .get("choices", [{}])[0]
    .get("message", {})
    .get("content", "No answer returned.")
  )


In [ ]:
#-------------For test cases------------! Dont touch
# #Native
# Native_results = retrieveNative(query=questions[0])
# Native_results

# #Faiss
# faiss_results = retrieveFaiss(query=questions[0])
# print('\nFaiss:')
# faiss_results

In [ ]:
def Ask_question(question):
    Native_results = retrieveNative(query=question)
    final_result = Native_results[0]['page_index']
    model = "meta/llama-3.2-11b-vision-instruct"
    nvidia_api_key = ''
    retrieved_image = images[final_result]
    Answer = RetriveAnswer_From_LLM(image=retrieved_image,
                                question=question,
                                model=model,
                                API_KEY=nvidia_api_key,
                                url="https://integrate.api.nvidia.com/v1/chat/completions")
    return Answer


In [ ]:
%%time
print(Ask_question('What does the encoder in Transformer consist of?'))